# Clasificación de sentimiento en reseñas — Etapa 1 (ML tradicional)

**Objetivo:** predecir si una reseña de producto en español es `positivo`, `negativo` o `neutral`, sin redes neuronales.
**Métrica:** accuracy.

**Modelo entregado (submission 012):** TF-IDF con canal posicional del discurso + tokens de regla discursiva basados en un léxico semántico, clasificados con `LinearSVC` (L1, C=2).

Este notebook documenta *por qué* ese es el modelo final: dónde se equivoca el modelo base, qué estructura de los datos decide la etiqueta, qué parte del error es irreducible y por qué el léxico no sobreajusta. Los cálculos pesados (validación cruzada) los ejecuta `src/experimento_016.py`; aquí solo se leen sus resultados.

In [ ]:
from pathlib import Path
import os, json
if not Path('src').exists():
    os.chdir('..')   # el notebook vive en notebooks/; se trabaja desde la raíz del proyecto
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay
from src.clauses import clauses, clean
from src.lex import polarity
from src.rulefeat import rule_tokens, ctype
from src.analisis_016 import opiniones

pd.set_option('display.max_colwidth', 200)
LABELS = ['negativo', 'neutral', 'positivo']
train = pd.read_csv('data/train.csv', dtype={'id': str})
ev = pd.read_csv('data/eval.csv', dtype={'id': str})
oof = pd.read_csv('reports/oof_016.csv', dtype={'id': str})
reg = json.loads(Path('reports/registro_016.json').read_text(encoding='utf-8'))
assert (oof.id == train.id).all()   # las predicciones OOF están en el mismo orden que train
print('train:', train.shape, '| eval:', ev.shape)

## 1. Los datos

In [ ]:
dist = train.label.value_counts().to_frame('n')
dist['%'] = (dist.n / len(train) * 100).round(1)
display(dist)
print(train.text.str.split().str.len().describe().round(1))
for lab in LABELS:
    print(f'\n--- {lab}:', train[train.label == lab].text.iloc[0])

## 2. Dónde se equivoca el modelo base

Predicciones *out-of-fold* (5 folds estratificados, semilla 0) del modelo 011, el mejor antes de esta ronda, y del modelo 012 final.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, m in zip(axes, ['011', '012']):
    acc = (oof.label == oof['pred_' + m]).mean()
    ConfusionMatrixDisplay.from_predictions(oof.label, oof['pred_' + m], labels=LABELS,
                                            ax=ax, colorbar=False)
    ax.set_title(f'Modelo {m} — accuracy OOF {acc:.4f}')
plt.tight_layout()
plt.show()

Neutral está prácticamente resuelto: casi todo el error es la confusión **positivo ↔ negativo**. La pregunta es qué hace difícil esa distinción.

## 3. Las frases de opinión no predicen la etiqueta

Si una reseña negativa fuera "una reseña con frases negativas", estas frases deberían concentrarse en una sola clase.

In [ ]:
texto = train.text.map(clean)
frases = ['error de compra', 'me decepciono', 'dolor de cabeza',
          'de maravilla', 'cinco estrellas', 'me alegra', 'ojos cerrados']
tabla = pd.DataFrame({f: train.label[texto.str.contains(f)].value_counts() for f in frases}).T
display(tabla.reindex(columns=LABELS).fillna(0).astype(int))

Las frases aparecen en ambas clases en proporciones parecidas. La razón es que casi todas las reseñas positivas y negativas son **mixtas**: elogian un aspecto y critican otro. Para analizarlas se parte cada reseña en cláusulas (`src/clauses.py`) y se estima la polaridad de cada una con un léxico semántico (`src/lex.py`).

In [ ]:
ejemplo = train.text.iloc[1]
print(ejemplo, '\n')
for k, (conector, cl) in enumerate(clauses(ejemplo)):
    print(f'{k}  conector={conector!r:16} polaridad={polarity(cl):+d} | {cl}')
print('\netiqueta real:', train.label.iloc[1])
print('tokens de regla:', list(rule_tokens(ejemplo)))

## 4. La estructura del discurso decide la etiqueta

Para cada reseña mixta (al menos una cláusula positiva y una negativa) se mira la **última** opinión: qué conector la introduce y si hay algo después de ella (un cierre). Se mide con qué frecuencia esa última opinión coincide con la etiqueta.

In [ ]:
filas = []
for text, label in zip(train.text, train.label):
    ops, n = opiniones(text)
    if len({p for *_, p in ops}) < 2:   # solo reseñas mixtas
        continue
    k, conector, _, p = ops[-1]
    filas.append({'conector': ctype(conector),
                  'cierre_despues': k < n - 1,
                  'gana_ultima': (p > 0) == (label == 'positivo')})
mixtas = pd.DataFrame(filas)
print(f'Reseñas mixtas detectadas: {len(mixtas)} ({len(mixtas) / len(train):.0%} del train)')
tabla = (mixtas.groupby(['conector', 'cierre_despues']).gana_ultima
               .agg(prop_gana_ultima='mean', n='size')
               .sort_values('n', ascending=False))
display(tabla.round(3))

La regla que emerge:

- Tras un conector **contrastivo** (*al final, aun así, con todo, pero, sin embargo*) la última opinión decide casi siempre.
- Sin conector, con *eso sí* o con *y*, decide en la gran mayoría de los casos, siempre que no haya un cierre después.
- Tras un conector **aditivo** (*por otro lado, de paso, por cierto, además*) o con un **cierre ambivalente** (*cada quien que juzgue, sigo con dudas...*) la proporción ronda el 50 %.

A esas últimas las llamamos **reseñas no predecibles**.

## 5. ¿Hay señal en las reseñas no predecibles?

Si el 50 % se debiera a que falta alguna feature, un clasificador entrenado *solo* con esas reseñas debería superar el azar. Cada palabra se marca con la polaridad de su cláusula (`P_`, `N_`) o como cierre (`T_`), para que el modelo pueda aprender, por ejemplo, que ciertos aspectos o intensidades pesan más.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

def tokens_polarizados(text):
    """Palabras marcadas con la polaridad de su cláusula, o T si van después de la última opinión."""
    ops, _ = opiniones(text)
    pol = {k: ('P' if p > 0 else 'N') for k, _, _, p in ops}
    ultima = ops[-1][0]
    toks = []
    for k, (_, cl) in enumerate(clauses(text)):
        tag = pol.get(k, 'T' if k > ultima else None)
        if tag:
            toks += [tag + '_' + w for w in cl.split()]
    return ' '.join(toks)

no_pred = train[oof.moneda & (train.label != 'neutral')]
X_np = no_pred.text.map(tokens_polarizados)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print(f'Reseñas no predecibles: {len(no_pred)} ({len(no_pred) / len(train):.1%} del train)')
print(f'Clase mayoritaria: {no_pred.label.value_counts(normalize=True).max():.3f}')
for C in [0.1, 0.3, 1.0]:
    modelo = make_pipeline(CountVectorizer(token_pattern=r'\S+', binary=True),
                           LogisticRegression(C=C, max_iter=3000))
    print(f'C={C}: accuracy CV {cross_val_score(modelo, X_np, no_pred.label, cv=cv).mean():.3f}')
cota = 1 - 0.5 * len(no_pred) / len(train)
print(f'Cota superior de accuracy si todo lo demás fuera perfecto: {cota:.3f}')

El clasificador no supera de forma apreciable a la clase mayoritaria: **en estas reseñas el texto no contiene información sobre la etiqueta**. Es error irreducible para cualquier modelo basado en el texto, clásico o profundo, y fija una cota superior de accuracy. La cota real es más baja: el léxico no detecta todas las cláusulas, así que hay reseñas no predecibles que aquí no se cuentan.

## 6. Modelo final y validación

`TextFeatures016(thirds=True, rules=True)` combina tres grupos de features:

- n-gramas globales (1–2);
- palabras etiquetadas por su posición en el discurso (`start/end` de cada bloque, tercio del texto, epílogo);
- **tokens de regla** (`R:`): polaridad de la última opinión × tipo de conector × si hay cierre después.

El modelo 011 es idéntico pero con `rules=False`. Ambos usan `LinearSVC(penalty='l1', C=2)`.

In [ ]:
folds = pd.read_csv('reports/folds_016.csv', dtype={'modelo': str})
resumen = folds.groupby('modelo')[['train_acc', 'val_acc']].agg(['mean', 'std'])
resumen['brecha'] = resumen[('train_acc', 'mean')] - resumen[('val_acc', 'mean')]
display(resumen.round(4))

pareado = folds.pivot(index='fold', columns='modelo', values='val_acc')
pareado['diferencia'] = pareado['012'] - pareado['011']
display(pareado.round(4))

In [ ]:
tipo = np.where(oof.moneda, 'no predecible', 'predecible')
aciertos = pd.DataFrame({'tipo': tipo,
                         '011': oof.label == oof.pred_011,
                         '012': oof.label == oof.pred_012})
display(aciertos.groupby('tipo').agg(n=('011', 'size'), acc_011=('011', 'mean'), acc_012=('012', 'mean')).round(4))

cambian = oof.pred_011 != oof.pred_012
print(f'Predicciones que cambian: {cambian.sum()} | en reseñas no predecibles: {(cambian & oof.moneda).sum()}')

En las reseñas predecibles las reglas mejoran de forma consistente. En las no predecibles solo intercambian aciertos por fallos, que es lo esperable si allí la etiqueta es aleatoria. Esto explica por qué la mejora global es pequeña y por qué no se refleja necesariamente en el leaderboard público.

## 7. Por qué el léxico no sobreajusta

`lex.py` no tiene parámetros: es una función fija, construida con semántica del español y no con las etiquetas del train. La pregunta es si la representación que produce le facilita al SVM memorizar. Se entrena cada modelo con todo el train y se compara cuántas features usa y cuánto acierta en train.

In [ ]:
comp = pd.DataFrame.from_dict({m: {k: v for k, v in d.items() if k != 'top_R'}
                               for m, d in reg['completos'].items()}, orient='index')
display(comp.round(4))
print('Tokens de regla con más peso en el modelo 012:')
display(pd.DataFrame(reg['completos']['012']['top_R']))

- Los tokens `R:` aparecen en cientos o miles de reseñas, así que su peso se estima con muchos ejemplos y no puede acomodarse a una reseña particular.
- Con penalización L1, una feature general que expresa la regla hace innecesarias muchas combinaciones específicas de n-gramas, y el optimizador las lleva a cero: el modelo 012 usa **menos** features.
- La accuracy de train en reseñas predecibles **baja** mientras la de validación **sube**: el modelo 011 memorizaba excepciones que no generalizaban.
- Las reseñas no predecibles se siguen memorizando en train con cualquiera de los dos modelos. Esa es la mayor parte de la brecha restante y no se corrige con features.

## 8. Producción y resultados en la competencia

In [ ]:
prod = json.loads(Path('reports/registro_produccion_012.json').read_text(encoding='utf-8'))
print('Tokens R: en el vocabulario final:', prod['n_rule_tokens_vocab'])
print('Predicciones distintas a la submission 011:', prod['n_changed_vs_011'])
print('Distribución de predicciones en eval:', prod['pred_distribution'])
publico = pd.DataFrame({'accuracy pública': {'011': 0.89888, '012': 0.89888}})
display(publico)

Ambas submissions empatan en el leaderboard público. Con unas 900 reseñas públicas, cada reseña vale ≈0.0011 puntos y el error estándar es de ≈±1 punto, así que el público no distingue modelos que difieren en decenas de reseñas. Se elige la 012 por la evidencia sobre las 12 000 reseñas del train: mejor validación y menor brecha.

## 9. Conclusiones y limitaciones

- La etiqueta de las reseñas mixtas depende de la **estructura del discurso** (última opinión y conector), no del vocabulario de opinión.
- Una parte del train tiene etiqueta **no predecible desde el texto**. Esto fija una cota superior de accuracy y explica la mayor parte de la brecha train–validación.
- Los tokens de regla mejoran el modelo en la parte predecible y lo hacen más simple (menos features activas).
- **Limitaciones:** el léxico es manual y tiene cobertura incompleta, en especial ante errores de tipeo y frases negativas poco frecuentes. Una alternativa sin intervención manual sería aprenderlo dentro de cada fold a partir de las cláusulas que siguen a conectores contrastivos.